In [ ]:
import os
import sys
import json
import shutil
import zipfile
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

In [ ]:
ROOT = Path("../../").resolve()
DB_DIR = ROOT / "db"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
METRICS_DIR = REPORTS_DIR / "metrics"

MIMICIV_DIR = DB_DIR / "mimiciv"
MIMICIV_ECG_DIR = DB_DIR / "mimiciv_ecg"
PROCESSED_DIR = DB_DIR / "processed"
LABELS_DIR = DB_DIR / "labels"

for d in [DB_DIR, MODELS_DIR, REPORTS_DIR, FIG_DIR, METRICS_DIR, MIMICIV_DIR, MIMICIV_ECG_DIR, PROCESSED_DIR, LABELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DB_DIR:", DB_DIR)
print("MIMICIV_DIR:", MIMICIV_DIR)
print("MIMICIV_ECG_DIR:", MIMICIV_ECG_DIR)

In [ ]:
MIMICIV_ECG_VERSION = "1.0"
MIMICIV_VERSION = "3.1"

DOWNLOAD_MIMICIV_ECG = True
DOWNLOAD_MIMICIV = True

ENCODER_PATH = MODELS_DIR / "encoder_pretrained.h5"
print("Encoder exists:", ENCODER_PATH.exists(), ENCODER_PATH)

In [ ]:
def run_cmd(cmd, cwd=None):
    result = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result.returncode

def ensure_file(path, desc="file"):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {desc}: {path}")
    print(f"Found {desc}: {path}")

In [ ]:
if DOWNLOAD_MIMICIV_ECG:
    cmd = f"wget -r -N -c -np https://physionet.org/files/mimic-iv-ecg/{MIMICIV_ECG_VERSION}/ -P {MIMICIV_ECG_DIR}"
    print(cmd)
    run_cmd(cmd)
else:
    print("Skipping ECG download in notebook; place files under db/mimiciv_ecg/")
if DOWNLOAD_MIMICIV:
    cmd = f"wget -r -N -c -np https://physionet.org/files/mimiciv/{MIMICIV_VERSION}/ -P {MIMICIV_DIR}"
    print(cmd)
    run_cmd(cmd)
else:
    print("Skipping MIMIC-IV download in notebook; place files under db/mimiciv/")

In [ ]:
ecg_candidates = list(MIMICIV_ECG_DIR.rglob("record_list.csv"))
print("record_list.csv files found:", ecg_candidates)

reports_candidates = list(MIMICIV_ECG_DIR.rglob("reports.csv"))
machine_candidates = list(MIMICIV_ECG_DIR.rglob("machine_measurements.csv"))
print("reports.csv:", reports_candidates)
print("machine_measurements.csv:", machine_candidates)

if not ecg_candidates:
    raise FileNotFoundError("Could not find record_list.csv in db/mimiciv_ecg/")

record_list_path = ecg_candidates[0]
ecg_index = pd.read_csv(record_list_path)
print(ecg_index.head())
print(ecg_index.columns.tolist())
print(ecg_index.shape)

ecg_index.columns = [c.lower() for c in ecg_index.columns]
ecg_index.head()

summary = {
    "n_ecg_rows": len(ecg_index),
    "n_unique_subjects": ecg_index["subject_id"].nunique() if "subject_id" in ecg_index.columns else None,
    "n_unique_studies": ecg_index["study_id"].nunique() if "study_id" in ecg_index.columns else None,
}
ecg_index_out = PROCESSED_DIR / "mimiciv_ecg_index.parquet"
ecg_index.to_parquet(ecg_index_out, index=False)
print("Saved:", ecg_index_out)
core_files = list(MIMICIV_DIR.rglob("patients.csv.gz")) + list(MIMICIV_DIR.rglob("patients.csv"))
hosp_adm_files = list(MIMICIV_DIR.rglob("admissions.csv.gz")) + list(MIMICIV_DIR.rglob("admissions.csv"))
icu_files = list(MIMICIV_DIR.rglob("icustays.csv.gz")) + list(MIMICIV_DIR.rglob("icustays.csv"))

print("patients:", core_files[:3])
print("admissions:", hosp_adm_files[:3])
print("icustays:", icu_files[:3])

In [ ]:
def read_csv_auto(path):
    path = Path(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)

patients_path = core_files[0] if core_files else None
admissions_path = hosp_adm_files[0] if hosp_adm_files else None
icustays_path = icu_files[0] if icu_files else None

if patients_path is None or admissions_path is None:
    raise FileNotFoundError("Need at least patients and admissions tables in db/mimiciv/")

patients = read_csv_auto(patients_path)
admissions = read_csv_auto(admissions_path)

print(patients.head())
print(admissions.head())
print(patients.shape, admissions.shape)
patients.columns = [c.lower() for c in patients.columns]
admissions.columns = [c.lower() for c in admissions.columns]

patients.head(), admissions.head()
for col in ["anchor_year_group", "dod"]:
    if col in patients.columns:
        print("patients has:", col)

for col in ["admittime", "dischtime", "deathtime", "edregtime", "edouttime"]:
    if col in admissions.columns:
        admissions[col] = pd.to_datetime(admissions[col], errors="coerce")

print(admissions[["subject_id", "hadm_id", "admittime", "dischtime"]].head())
link = ecg_index.copy()

if "subject_id" not in link.columns:
    raise KeyError("record_list.csv must include subject_id.")

subject_summary = (
    link.groupby("subject_id")
    .agg(
        n_ecg=("study_id", "count") if "study_id" in link.columns else ("subject_id", "count"),
        first_study=("study_id", "min") if "study_id" in link.columns else ("subject_id", "min"),
        last_study=("study_id", "max") if "study_id" in link.columns else ("subject_id", "max"),
    )
    .reset_index()
)

subject_summary.head()

In [ ]:
subject_summary = subject_summary.merge(
    patients[[c for c in ["subject_id", "gender", "anchor_age", "anchor_year", "anchor_year_group", "dod"] if c in patients.columns]],
    on="subject_id",
    how="left",
)

subject_summary.head()
adm_cols = [c for c in ["subject_id", "hadm_id", "admittime", "dischtime", "deathtime", "admission_type", "admission_location", "discharge_location", "insurance", "language", "marital_status", "race"] if c in admissions.columns]
adm = admissions[adm_cols].copy()
adm.head()
adm_summary = (
    adm.groupby("subject_id")
    .agg(
        n_admissions=("hadm_id", "nunique") if "hadm_id" in adm.columns else ("subject_id", "count"),
        first_admit=("admittime", "min") if "admittime" in adm.columns else ("subject_id", "min"),
        last_admit=("admittime", "max") if "admittime" in adm.columns else ("subject_id", "max"),
    )
    .reset_index()
)

subject_summary = subject_summary.merge(adm_summary, on="subject_id", how="left")
subject_summary.head()
subject_index_out = PROCESSED_DIR / "subject_ecg_index.parquet"
subject_summary.to_parquet(subject_index_out, index=False)
print("Saved:", subject_index_out)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
subject_summary["n_ecg"].fillna(0).hist(bins=50)
plt.title("ECGs per subject")
plt.xlabel("count")

plt.subplot(1, 2, 2)
subject_summary["n_admissions"].fillna(0).hist(bins=50)
plt.title("Admissions per subject")
plt.xlabel("count")

plt.tight_layout()
fig_path = FIG_DIR / "subject_summary_hist.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("Saved:", fig_path)

In [ ]:
ensure_file(ENCODER_PATH, "pretrained encoder")
config = {
    "root": str(ROOT),
    "db_dir": str(DB_DIR),
    "mimiciv_dir": str(MIMICIV_DIR),
    "mimiciv_ecg_dir": str(MIMICIV_ECG_DIR),
    "processed_dir": str(PROCESSED_DIR),
    "labels_dir": str(LABELS_DIR),
    "models_dir": str(MODELS_DIR),
    "encoder_path": str(ENCODER_PATH),
}
with open(PROCESSED_DIR / "pipeline_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved pipeline_config.json")